In [6]:
# 4.计算每个店铺与某一点（广东财经大学）的距离，并进行归类。对店铺不同距离类别进行统计

import pandas as pd
import math
from scipy.spatial.distance import cdist
    
#cdist：来自 scipy.spatial.distance，用于计算两个集合中所有点的成对距离。这里会用它来计算两点之间的欧氏距离。


In [7]:
'''读取数据并处理坐标'''

data=pd.read_excel(r'C:\Users\11707\Desktop\vspython\高德周边.xlsx', 'Sheet1')
poi_r=data['newlocation'].str.split(',')

'''
data['newlocation']：Excel 中有一列 newlocation，格式是 "经度,纬度"。   
.str.split(',')：将每个位置字符串按逗号拆分成列表，例如 "113.35,23.09" → ['113.35', '23.09']。'''


poi=[]
for r in poi_r:
    r1=[float(item) for item in r]
    poi.append(r1)

'''
遍历每一行的 poi_r（每个经纬度列表）。
[float(item) for item in r]：把字符串转换为浮点数。
poi.append(r1)：最终得到一个二维列表 poi，每个元素是 [经度, 纬度] 的浮点数列表。
'''


'\n遍历每一行的 poi_r（每个经纬度列表）。\n[float(item) for item in r]：把字符串转换为浮点数。\npoi.append(r1)：最终得到一个二维列表 poi，每个元素是 [经度, 纬度] 的浮点数列表。\n'

In [8]:
'''定义距离函数'''
def distance(l1,l2):
    dis=cdist([l1],[l2],metric='euclidean')
    dis=list(list(list(dis))[0]*100000)[0]
    
    if dis<1000:
        cla='1公里'
    elif 1000<=dis<2000:
        cla='2公里'
    elif 2000<=dis<3000:
        cla='3公里'
    elif 3000<=dis<4000:
        cla='4公里'
    else:
        cla='5公里'
    return [dis,cla]

'''
l1 和 l2 是两个点的经纬度列表，格式 [纬度, 经度] 或 [经度, 纬度]（后续代码里注意顺序）。
cdist([l1],[l2],metric='euclidean')：计算欧氏距离。
[l1] 和 [l2] 需要用列表包裹成二维数组。返回结果是一个二维数组，形状 (1,1)。

dis=list(list(list(dis))[0]*100000)[0]：
先把 cdist 返回的二维数组展开成标量。
乘以 100000：大概是为了把经纬度的度转换成近似米（粗略换算：1°纬度≈111km，1°经度随纬度变化）。

根据距离将每个点分类：
<1000 → “1公里”
1000~2000 → “2公里”
2000~3000 → “3公里”
3000~4000 → “4公里”
>=4000 → “5公里”
返回一个列表 [距离值, 分类]。
'''

"\nl1 和 l2 是两个点的经纬度列表，格式 [纬度, 经度] 或 [经度, 纬度]（后续代码里注意顺序）。\ncdist([l1],[l2],metric='euclidean')：计算欧氏距离。\n[l1] 和 [l2] 需要用列表包裹成二维数组。返回结果是一个二维数组，形状 (1,1)。\n\ndis=list(list(list(dis))[0]*100000)[0]：\n先把 cdist 返回的二维数组展开成标量。\n乘以 100000：大概是为了把经纬度的度转换成近似米（粗略换算：1°纬度≈111km，1°经度随纬度变化）。\n\n根据距离将每个点分类：\n<1000 → “1公里”\n1000~2000 → “2公里”\n2000~3000 → “3公里”\n3000~4000 → “4公里”\n>=4000 → “5公里”\n返回一个列表 [距离值, 分类]。\n"

In [9]:
'''计算所有点到目标点的距离,保存距离及其分类到Excel表中'''

point=[23.090164,113.354053]
dis=[]
cla=[]
for p in poi:
    p1=distance(point,p[::-1])
    dis.append(p1[0])
    cla.append(p1[1])

data['距离']=dis
data['类别']=cla
data_1=data.loc[:,['距离','类别']]

'''
p[::-1]：将 [经度, 纬度] 反转成 [纬度, 经度]，保证和 point 顺序一致。
distance(point, p[::-1])：计算距离和分类。
dist.append(p1[0])：保存距离。
clify.append(p1[1])：保存分类。
'''

'\np[::-1]：将 [经度, 纬度] 反转成 [纬度, 经度]，保证和 point 顺序一致。\ndistance(point, p[::-1])：计算距离和分类。\ndist.append(p1[0])：保存距离。\nclify.append(p1[1])：保存分类。\n'

In [10]:
'''统计各类别数量,绘制柱状'''

from pyecharts import options as opts
from pyecharts.charts import Bar

clify_count=data_1['类别'].value_counts()
print(clify_count)

c = (
    Bar()
    .add_xaxis(clify_count.index.values.tolist())
    .add_yaxis("不同距离店铺数", clify_count.values.tolist())
    .set_global_opts(title_opts=opts.TitleOpts(title="POI点类别分布", subtitle="按距离分类图"))
    .render(r'C:\Users\11707\Desktop\vspython\按距离分类图.html'))


类别
5公里    72
4公里    66
3公里    40
1公里    30
2公里    17
Name: count, dtype: int64
